# 06 FHIR Condition Silver — Silver Layer

Read the `condition` Bronze Delta table, parse the original FHIR JSON, clean and deduplicate records, save the Silver table, and maintain SCD Type 2 history.

**Prerequisite:** Run the matching `Condition` ingestion notebook first and attach `FHIR_Lakehouse`.

**Bronze design:** The Bronze table stores the complete FHIR resource in `resource_json` plus extraction/load metadata. This notebook parses the JSON before accessing resource fields.

In [1]:
from pyspark.sql.functions import (
    col, lit, trim, lower, when, row_number,
    sha2, coalesce, from_json, get_json_object,
    current_timestamp
)
from pyspark.sql.window import Window
from delta.tables import DeltaTable

# READ BRONZE DATA
bronze_condition_df = spark.read.table("bronze.condition")

print("Bronze Condition loaded!")
print("Bronze records:", bronze_condition_df.count())
print("Bronze columns:", bronze_condition_df.columns)

StatementMeta(, 8bb3145f-82dc-4a17-8cb0-efd80b1958ef, 3, Finished, Available, Finished, False)

Bronze Condition loaded!
Bronze records: 11878
Bronze columns: ['abatementDateTime', 'asserter', 'bodySite', 'category', 'clinicalStatus', 'code', 'encounter', 'evidence', 'extension', 'id', 'identifier', 'meta', 'note', 'onsetDateTime', 'onsetString', 'recordedDate', 'recorder', 'resourceType', 'severity', 'stage', 'subject', 'text', 'verificationStatus', 'resource_json', 'extraction_timestamp', 'api_url_or_params', 'load_timestamp', 'source_file']


In [2]:
# PARSE THE ORIGINAL FHIR JSON

# Infer the resource schema from the JSON records in Bronze.
resource_schema = spark.read.json(
    bronze_condition_df.select("resource_json").rdd.map(lambda row: row["resource_json"])
).schema

silver_condition_df = (
    bronze_condition_df
    .withColumn(
        "resource",
        from_json(col("resource_json"), resource_schema)
    )
    .select(
        "resource.*",
        "resource_json",
        "extraction_timestamp",
        "api_url_or_params",
        "load_timestamp",
        "source_file"
    )
)

print("FHIR JSON parsed into resource columns.")
silver_condition_df.printSchema()

StatementMeta(, 8bb3145f-82dc-4a17-8cb0-efd80b1958ef, 4, Finished, Available, Finished, False)

FHIR JSON parsed into resource columns.
root
 |-- abatementDateTime: string (nullable = true)
 |-- asserter: struct (nullable = true)
 |    |-- display: string (nullable = true)
 |    |-- reference: string (nullable = true)
 |-- bodySite: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- coding: array (nullable = true)
 |    |    |    |-- element: struct (containsNull = true)
 |    |    |    |    |-- code: string (nullable = true)
 |    |    |    |    |-- display: string (nullable = true)
 |    |    |    |    |-- system: string (nullable = true)
 |-- category: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- coding: array (nullable = true)
 |    |    |    |-- element: struct (containsNull = true)
 |    |    |    |    |-- code: string (nullable = true)
 |    |    |    |    |-- display: string (nullable = true)
 |    |    |    |    |-- system: string (nullable = true)
 |    |    |-- text: string (nullable = true)


In [3]:
# CLEAN CONDITION DATA

silver_condition_df = silver_condition_df.filter(
    col("id").isNotNull() & (trim(col("id")) != "")
).filter(col("resourceType") == "Condition")

# Extract the first clinical status coding when present.
silver_condition_df = (
    silver_condition_df
    .withColumn(
        "clinical_status",
        lower(trim(get_json_object("resource_json", "$.clinicalStatus.coding[0].code")))
    )
    .withColumn(
        "clinical_status_display",
        get_json_object("resource_json", "$.clinicalStatus.coding[0].display")
    )
)

print("Condition cleaning completed!")

StatementMeta(, 8bb3145f-82dc-4a17-8cb0-efd80b1958ef, 5, Finished, Available, Finished, False)

Condition cleaning completed!


In [4]:
# DEDUPLICATE BY RESOURCE ID

window_spec = (
    Window
    .partitionBy("id")
    .orderBy(
        col("extraction_timestamp").desc(),
        col("load_timestamp").desc()
    )
)

silver_condition_df = (
    silver_condition_df
    .withColumn("row_number", row_number().over(window_spec))
    .filter(col("row_number") == 1)
    .drop("row_number")
)

print("Bronze records:", bronze_condition_df.count())
print("Silver unique records:", silver_condition_df.count())

StatementMeta(, 8bb3145f-82dc-4a17-8cb0-efd80b1958ef, 6, Finished, Available, Finished, False)

Bronze records: 11878
Silver unique records: 11878


In [5]:
# SAVE SILVER TABLE

spark.sql("CREATE SCHEMA IF NOT EXISTS silver")

(
    silver_condition_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.condition")
)

print("Saved silver.condition")
print("Silver records:", spark.table("silver.condition").count())

StatementMeta(, 8bb3145f-82dc-4a17-8cb0-efd80b1958ef, 7, Finished, Available, Finished, False)

Saved silver.condition
Silver records: 11878


In [6]:
# INITIALIZE SCD TYPE 2 HISTORY IF IT DOES NOT EXIST

history_table = "silver.condition_history"

if not spark.catalog.tableExists(history_table):
    initial_history_df = (
        spark.table("silver.condition")
        .withColumn(
            "record_hash",
            sha2(coalesce(col("resource_json"), lit("")), 256)
        )
        .withColumn(
            "effective_start_date",
            col("extraction_timestamp").cast("timestamp")
        )
        .withColumn(
            "effective_end_date",
            lit(None).cast("timestamp")
        )
        .withColumn("is_current", lit(True))
    )

    (
        initial_history_df.write
        .format("delta")
        .mode("errorifexists")
        .saveAsTable(history_table)
    )

    print("Initial SCD Type 2 history created:", history_table)
else:
    print("History table already exists; existing history will be preserved.")

history_df = spark.table(history_table)
print("History records:", history_df.count())
history_df.groupBy("is_current").count().show()

StatementMeta(, 8bb3145f-82dc-4a17-8cb0-efd80b1958ef, 8, Finished, Available, Finished, False)

History table already exists; existing history will be preserved.
History records: 13450
+----------+-----+
|is_current|count|
+----------+-----+
|      true|13450|
+----------+-----+



In [7]:
# DETECT NEW OR CHANGED RECORDS

incoming_df = (
    spark.table("silver.condition")
    .withColumn(
        "record_hash",
        sha2(coalesce(col("resource_json"), lit("")), 256)
    )
)

current_history_df = (
    spark.table(history_table)
    .filter(col("is_current") == True)
    .select("id", "record_hash")
)

comparison_df = (
    incoming_df.alias("src")
    .join(current_history_df.alias("tgt"), on="id", how="left")
)

changed_or_new_df = comparison_df.filter(
    col("tgt.record_hash").isNull() |
    (col("src.record_hash") != col("tgt.record_hash"))
)

new_versions_df = changed_or_new_df.select("src.*")

print("Incoming records:", incoming_df.count())
print("Existing current records:", current_history_df.count())
print("New or changed records:", new_versions_df.count())

StatementMeta(, 8bb3145f-82dc-4a17-8cb0-efd80b1958ef, 9, Finished, Available, Finished, False)

Incoming records: 11878
Existing current records: 13450
New or changed records: 4


In [8]:
# APPLY SCD TYPE 2 CHANGES

changed_ids_df = new_versions_df.select("id").distinct()
changed_count = new_versions_df.count()

if changed_count > 0:
    run_timestamp = spark.sql(
        "SELECT current_timestamp() AS ts"
    ).first()["ts"]

    delta_history = DeltaTable.forName(spark, history_table)

    (
        delta_history.alias("tgt")
        .merge(
            changed_ids_df.alias("src"),
            "tgt.id = src.id AND tgt.is_current = true"
        )
        .whenMatchedUpdate(set={
            "is_current": lit(False),
            "effective_end_date": lit(run_timestamp).cast("timestamp")
        })
        .execute()
    )

    new_history_df = (
        new_versions_df
        .withColumn(
            "effective_start_date",
            lit(run_timestamp).cast("timestamp")
        )
        .withColumn(
            "effective_end_date",
            lit(None).cast("timestamp")
        )
        .withColumn("is_current", lit(True))
    )

    (
        new_history_df.write
        .format("delta")
        .mode("append")
        .option("mergeSchema", "true")
        .saveAsTable(history_table)
    )

    print("SCD Type 2 changes applied:", changed_count)
else:
    print("No new or changed records. History remains unchanged.")

StatementMeta(, 8bb3145f-82dc-4a17-8cb0-efd80b1958ef, 10, Finished, Available, Finished, False)

SCD Type 2 changes applied: 4


In [9]:
# VALIDATE HISTORY TABLE

history_df = spark.table(history_table)

print("Total history records:", history_df.count())
history_df.groupBy("is_current").count().show()

duplicate_current_ids = (
    history_df
    .filter(col("is_current") == True)
    .groupBy("id")
    .count()
    .filter(col("count") > 1)
)

print("IDs with more than one current version:", duplicate_current_ids.count())
duplicate_current_ids.show(20, truncate=False)

StatementMeta(, 8bb3145f-82dc-4a17-8cb0-efd80b1958ef, 11, Finished, Available, Finished, False)

Total history records: 13454
+----------+-----+
|is_current|count|
+----------+-----+
|      true|13454|
+----------+-----+

IDs with more than one current version: 0
+---+-----+
|id |count|
+---+-----+
+---+-----+



In [10]:
# FINAL SILVER VALIDATION

spark.table("silver.condition").select(
    "id", "resourceType", "extraction_timestamp", "load_timestamp"
).show(10, truncate=False)

StatementMeta(, 8bb3145f-82dc-4a17-8cb0-efd80b1958ef, 12, Finished, Available, Finished, False)

+------+------------+--------------------------+--------------------------+
|id    |resourceType|extraction_timestamp      |load_timestamp            |
+------+------------+--------------------------+--------------------------+
|492268|Condition   |2026-11-18 19:14:31.222807|2026-11-18 19:16:20.299521|
|820970|Condition   |2026-11-18 19:14:31.222807|2026-11-18 19:16:20.299521|
|821118|Condition   |2026-11-18 19:14:31.222807|2026-11-18 19:16:20.299521|
|821137|Condition   |2026-11-18 19:14:31.222807|2026-11-18 19:16:20.299521|
|821321|Condition   |2026-11-18 19:14:31.222807|2026-11-18 19:16:20.299521|
|821325|Condition   |2026-11-18 19:14:31.222807|2026-11-18 19:16:20.299521|
|821334|Condition   |2026-11-18 19:14:31.222807|2026-11-18 19:16:20.299521|
|821343|Condition   |2026-11-18 19:14:31.222807|2026-11-18 19:16:20.299521|
|822300|Condition   |2026-11-18 19:14:31.222807|2026-11-18 19:16:20.299521|
|822305|Condition   |2026-11-18 19:14:31.222807|2026-11-18 19:16:20.299521|
+------+----